In [1]:
import os
import random
import shutil

# Validation Flow

In [2]:
def generate_validation_dataset(
    src_dataset_path: str,
    output_path: str,
    num_images: int = 10,
    seed: int = 42,
) -> None:
    """
    Generate a validation dataset from the MVTec_AD bottle category.

    Randomly selects ``num_images`` good images from the bottle train split,
    copies them to ``output_path/test/good/``, and (TODO) produces augmented
    defective variants alongside their ground-truth masks.

    Output layout:
        output_path/
            ground_truth/   # defect masks (populated once augmentation is implemented)
            test/
                good/       # unmodified good images
                <defect>/   # augmented defective images (TODO)

    Args:
        src_dataset_path: Path to the root of the MVTec_AD dataset
                          (the folder that contains the ``bottle`` subfolder).
        output_path:      Destination folder for the generated validation dataset.
        num_images:       Number of good images to sample.
        seed:             Random seed for reproducibility.
    """
    good_images_dir = os.path.join(src_dataset_path, "bottle", "train", "good")
    if not os.path.isdir(good_images_dir):
        raise FileNotFoundError(
            f"Good images directory not found: {good_images_dir}"
        )

    all_images = sorted(
        f for f in os.listdir(good_images_dir)
        if os.path.isfile(os.path.join(good_images_dir, f))
    )
    if not all_images:
        raise ValueError(f"No images found in {good_images_dir}")

    rng = random.Random(seed)
    selected = rng.sample(all_images, min(num_images, len(all_images)))

    # Create output directory structure
    test_good_dir = os.path.join(output_path, "test", "good")
    ground_truth_dir = os.path.join(output_path, "ground_truth")
    os.makedirs(test_good_dir, exist_ok=True)
    os.makedirs(ground_truth_dir, exist_ok=True)

    # Copy selected good images to test/good/
    for filename in selected:
        src = os.path.join(good_images_dir, filename)
        dst = os.path.join(test_good_dir, filename)
        shutil.copy2(src, dst)

    print(
        f"Validation dataset created at '{output_path}' "
        f"with {len(selected)} good image(s) (seed={seed})."
    )

    # TODO: Augment selected images to simulate defects.
    #       For each augmented image:
    #         - save the defective image under test/<defect_type>/
    #         - save the corresponding binary mask under ground_truth/<defect_type>/

In [5]:
generate_validation_dataset(
    src_dataset_path="../data/datasets/MVTec_AD",
    output_path="../data/datasets/validation_dataset",
    num_images=10,
    seed=42
)

Validation dataset created at '../data/datasets/validation_dataset' with 10 good image(s) (seed=42).
